# Building a RAG Pipeline - Nexus-LLM

This notebook shows how to build a Retrieval-Augmented Generation (RAG) pipeline that combines document retrieval with LLM generation for accurate, grounded responses.

## What is RAG?

RAG enhances LLM responses by first retrieving relevant documents from a knowledge base, then conditioning the model's generation on the retrieved context.

## 1. Set Up the Embedding Engine

In [ ]:
from nexus_llm.rag import EmbeddingEngine

embedder = EmbeddingEngine(
    model_name="nexus-embedding-large",
    device="auto",
    batch_size=32,
    normalize_embeddings=True,
)

# Test the embedder
test_embedding = embedder.embed("Hello, world!")
print(f"Embedding dimension: {test_embedding.shape[0]}")
print(f"Embedding norm: {sum(x**2 for x in test_embedding)**0.5:.4f}")

## 2. Ingest and Chunk Documents

In [ ]:
from nexus_llm.rag import DocumentStore, ChunkingStrategy

# Configure chunking
chunking = ChunkingStrategy(
    method="recursive",
    chunk_size=512,            # Target chunk size in tokens
    chunk_overlap=64,          # Overlap between chunks
    separators=["\n\n", "\n", ". ", " ", ""],
)

# Create document store
doc_store = DocumentStore(
    storage_path="./data/doc_store",
    chunking_strategy=chunking,
)

# Ingest documents
doc_ids = doc_store.ingest_files([
    "./docs/product_manual.pdf",
    "./docs/api_reference.html",
    "./docs/faq.md",
])

print(f"Ingested {len(doc_ids)} documents")

# Also add text directly
doc_store.ingest_text(
    text="Nexus-LLM supports 128K context length with Flash Attention 2. "
         "The system is compatible with NVIDIA A100, H100, and consumer GPUs.",
    metadata={"source": "knowledge_base", "topic": "specs"},
)

# View chunk statistics
chunks = doc_store.get_all_chunks()
print(f"Total chunks: {len(chunks)}")
print(f"Average chunk length: {sum(len(c.text) for c in chunks)/len(chunks):.0f} chars")

## 3. Build the Vector Index

In [ ]:
from nexus_llm.rag import VectorIndex

vector_index = VectorIndex(
    embedding_engine=embedder,
    index_type="faiss",       # Options: faiss, hnswlib, chromadb
    metric="cosine",
    dimension=1024,
    index_path="./data/vector_index",
)

# Build the index from document chunks
vector_index.build(chunks)
print(f"Vector index built with {vector_index.size} entries")

# Test a search
results = vector_index.search("context length", top_k=3)
for rank, result in enumerate(results, 1):
    print(f"\n{rank}. [Score: {result.score:.3f}] {result.text[:150]}...")

## 4. Create the RAG Pipeline

In [ ]:
from nexus_llm import InferenceEngine
from nexus_llm.rag import RAGPipeline, RetrievalConfig

engine = InferenceEngine(model_name="nexus-7b-chat", device="auto")

retrieval_config = RetrievalConfig(
    top_k=5,                      # Retrieve top 5 chunks
    similarity_threshold=0.65,    # Minimum similarity
    reranking=True,               # Cross-encoder reranking
    rerank_top_k=3,               # Keep top 3 after reranking
    max_context_tokens=2048,      # Max context from retrieval
)

pipeline = RAGPipeline(
    inference_engine=engine,
    document_store=doc_store,
    vector_index=vector_index,
    retrieval_config=retrieval_config,
)

print("RAG pipeline initialized!")

## 5. Query the Pipeline

In [ ]:
# Ask a question
result = pipeline.query("What GPUs does Nexus-LLM support?")

print(f"Answer: {result.answer}")
print(f"\nSources:")
for src in result.sources:
    print(f"  [{src.score:.3f}] {src.document_id} - chunk {src.chunk_index}")
    print(f"       {src.text[:100]}...")

In [ ]:
# Ask another question
result = pipeline.query("How do I configure the server?")

print(f"Answer: {result.answer}")
print(f"\nConfidence: {result.confidence:.2f}")
print(f"Retrieved chunks: {len(result.sources)}")

## 6. Adding Documents Dynamically

In [ ]:
# Add new documents without rebuilding the entire index
new_doc_id = doc_store.ingest_text(
    text="The latest version of Nexus-LLM (v2.1) adds support for Mamba-based "
         "architectures and improved quantization with GPTQ and AWQ methods.",
    metadata={"source": "release_notes", "version": "2.1"},
)

# Incrementally update the vector index
new_chunks = doc_store.get_chunks(new_doc_id)
vector_index.add(new_chunks)
print(f"Added {len(new_chunks)} new chunks to the index")

# Query the updated knowledge base
result = pipeline.query("What quantization methods are supported?")
print(f"\nAnswer: {result.answer}")

## 7. Evaluation

In [ ]:
from nexus_llm.rag import RAGEvaluator

evaluator = RAGEvaluator()

# Evaluate retrieval quality
retrieval_metrics = evaluator.evaluate_retrieval(
    pipeline=pipeline,
    qa_pairs=[
        {"question": "What GPUs are supported?", "relevant_doc_ids": ["doc_001"]},
        {"question": "How to configure the server?", "relevant_doc_ids": ["doc_002"]},
    ],
    metrics=["precision@k", "recall@k", "mrr", "ndcg"],
)

print("Retrieval Metrics:")
for metric, value in retrieval_metrics.items():
    print(f"  {metric}: {value:.3f}")

# Evaluate end-to-end generation quality
gen_metrics = evaluator.evaluate_generation(
    pipeline=pipeline,
    qa_pairs=[
        {"question": "What is the max context length?", "answer": "128K tokens"},
    ],
    metrics=["faithfulness", "relevancy", "completeness"],
)

print("\nGeneration Metrics:")
for metric, value in gen_metrics.items():
    print(f"  {metric}: {value:.3f}")